# Lone-Wolf-Howler — guarded retraining

This notebook checks the source and output-suppression tests, stages only the approved JSONL files, and runs the recovery-aware training pipeline. GPU and Internet must be enabled. Add a Kaggle Secret named `HF_TOKEN` with a Hugging Face token that can publish to your configured model repository. Do not paste tokens into notebook cells.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'Shadow-ronin-master'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/yalangisexter-ux/Shadow-ronin-master.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)

# Require the publishing secret before spending GPU time.
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        print(f"[warning] Kaggle UserSecretsClient could not retrieve HF_TOKEN: {exc}")

if not hf_token:
    raise RuntimeError('HF_TOKEN not found! Either attach HF_TOKEN in Kaggle Notebook > Add-ons > Secrets, or configure secret HF_TOKEN in GitHub repo.')
os.environ['HF_TOKEN'] = hf_token

# Stage only approved datasets from repository files or attached Kaggle inputs.
# Deliberately exclude Dark_web.jsonl and Telugu_uncensored.jsonl.
DATA = WORK / 'lone_wolf_training_data'
DATA.mkdir(parents=True, exist_ok=True)
approved = ('combined_clean_dataset_augmented.jsonl', 'combined_clean_dataset.jsonl')
found = []
search_roots = [REPO, Path('/kaggle/input')]
for name in approved:
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob(name))
    if candidates:
        src = candidates[0]
        dst = DATA / name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        found.append(name)
if not found:
    raise FileNotFoundError(
        'No approved training data found. Attach a Kaggle dataset containing '
        'combined_clean_dataset_augmented.jsonl and/or combined_clean_dataset.jsonl.'
    )
print(f"[dataset] staged approved files: {found}")

os.environ['TRAINING_DATA_DIR'] = str(DATA)
os.environ['FORCE_SCRATCH_TRAINING'] = '1'
os.environ['ENABLE_GRPO'] = '0'
os.environ['DISABLE_RAG'] = '1'  # RAG remains disabled for this run.
os.environ.setdefault('MODEL_ID', 'Qwen/Qwen3-1.7B')

# Syntax and output-regression checks before training.
for name in ('Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py',
             'Lone-Wolf-Howler-End-to-End-Recovery-First-Test.py',
             'Shadow-Ronin-4B-End-to-End-Recovery-First-Master.py',
             'lone_wolf_inference.py'):
    subprocess.run([sys.executable, '-m', 'py_compile', str(REPO / name)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'], check=True)

# Ensure bitsandbytes & accelerate are installed in this environment
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.43.0', 'accelerate', 'peft'], check=False)

# Run the fixed pipeline with automatic restart on environment repair (code 75)
max_attempts = 3
pipeline_completed = False
for attempt in range(1, max_attempts + 1):
    print(f"[pipeline] Starting pipeline execution (attempt {attempt}/{max_attempts})...")
    proc = subprocess.run(
        [sys.executable, str(REPO / 'Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py')],
        cwd=str(REPO),
        env=os.environ.copy()
    )
    if proc.returncode == 75:
        print("[pipeline] Environment repaired; restarting in fresh process...")
        continue
    elif proc.returncode != 0:
        raise RuntimeError(f"Pipeline failed with exit code {proc.returncode}")
    else:
        print("[pipeline] Process exited with code 0; inspect the final stage and upload logs to verify artifacts.")
        pipeline_completed = True
        break

if not pipeline_completed:
    raise RuntimeError(f"Pipeline did not complete after {max_attempts} attempts; restart was repeatedly requested (exit code 75). Review the bitsandbytes preflight logs.")
